# RP 单任务：API planner + DDM
Kaggle 开启 GPU 和 Internet，把 `rp_code.zip` 添加为输入数据。在 Secrets 中添加 `DISCOVERY_API_KEY` 并授权此 Notebook，然后依次运行下面的单元。单任务使用一张 T4。

In [ ]:
import os
import sys
import subprocess
import zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ.update(
    DISCOVERY_API_BASE='https://discovery-api.intern-ai.org.cn/v1',
    DISCOVERY_API_KEY=UserSecretsClient().get_secret('DISCOVERY_API_KEY'),
    DISCOVERY_MODEL='deepseek-v4.1-flash-1017',
    MUJOCO_GL='egl',
    PYOPENGL_PLATFORM='egl',
    CUDA_VISIBLE_DEVICES='0',
)
archive = next(Path('/kaggle/input').rglob('rp_code.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/kaggle/working')
repo = Path('/kaggle/working/reflect-vlm')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'mujoco==3.1.2', 'gym==0.26.2', 'numpy<2',
    'wandb==0.16.2', 'setuptools<81', 'cloudpickle',
    'absl-py', 'ml_collections', 'pandas', 'Pillow', 'imageio', 'imageio-ffmpeg', 'openai',
    'diffusers>=0.30,<1', 'transformers>=4.44,<5', 'accelerate', 'safetensors',
], check=True)


In [ ]:
out = Path('/kaggle/working/rp_diffusion')
command = [
    'bash', str(repo / 'scripts/eval_reflect_vlm.sh'), 'diffusion',
    '--max_steps=10',
    f'--save_dir={out}',
    '--logging.output_dir=/kaggle/working/wandb',
]
with open('/kaggle/working/rp_diffusion.log', 'w') as log:
    process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    returncode = process.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


In [ ]:
import csv
from IPython.display import Image, display

with open(out / 'meta.csv') as f:
    for row in csv.DictReader(f):
        print(f"Step {row['step_id']}: {row['agent_action']} -> "
              f"{row['agent_action_revised']} -> 执行 {row['action_description']}")
traj = out / '0'
for label, path in [
    ('当前图', traj / '0.png'),
    ('DDM 预测图', next(traj.glob('gen-0-*.png'))),
    ('实际执行后', traj / '1.png'),
]:
    print(label)
    display(Image(filename=str(path)))
